In [0]:
import json
import numpy as np
import re
from pyspark.ml import PipelineModel
from pyspark.ml.feature import Tokenizer, StopWordsRemover
from pyspark.ml.clustering import LocalLDAModel
 
CATALOGO        = "proyecto_bda"
SCHEMA          = "bda_schema"
VOLUMEN_MODELOS = f"/Volumes/{CATALOGO}/{SCHEMA}/bda_modelos"
 
NOMBRES_CLUSTERS = {
    0: "Decepcionado silencioso",
    1: "Recurrente satisfecho",
    2: "Validador masivo",
    3: "Crítico detallista",
    4: "Foodie activo",
    5: "Promotor genuino",
    6: "Experiencia errática",
}
NOMBRES_TOPICOS = {
    0: "Experiencia general",
    1: "Pollo a la brasa",
    2: "Calidad-precio",
    3: "Recomendación positiva",
    4: "Ceviche y platos bandera",
}
SPANISH_STOPS = [
    'de','la','que','el','en','y','a','los','del','se','las','por','un','para',
    'con','no','una','su','al','lo','como','más','pero','sus','le','ya','o',
    'este','sí','porque','esta','entre','cuando','muy','sin','sobre','también',
    'me','hasta','hay','donde','quien','desde','todo','nos','durante','todos',
    'uno','les','ni','contra','otros','ese','eso','ante','ellos','e','esto',
    'mí','antes','algunos','qué','unos','yo','otro','otras','otra','él','tanto',
    'esa','estos','mucho','quienes','nada','muchos','cual','poco','ella','estar',
    'estas','algunas','algo','nosotros','mi','mis','tú','te','ti','tu','tus',
    'ellas','nos','vosotros','vosotras','os','mío','mía','míos','mías','tuyo',
    'si','lugar','sitio','vez','mas','ser','hacer','ir','muy','bien','todo',
    'nada','así','aquí','café','restaurante','local','comida','lima','peru',
]

In [0]:
dbutils.widgets.text("modo",               "kmeans")  # "kmeans" | "lda"
dbutils.widgets.text("log_review_count",   "2.0")
dbutils.widgets.text("avg_rating",         "4.0")
dbutils.widgets.text("std_rating",         "0.5")
dbutils.widgets.text("log_avg_word_count", "3.0")
dbutils.widgets.text("texto",              "")
 
modo = dbutils.widgets.get("modo")

In [0]:
# ── Rama KMeans ───────────────────────────────────────────────
# Carga el PipelineModel completo (VectorAssembler → StandardScaler → KMeans)
# exactamente como fue guardado en el notebook de entrenamiento.
# La inferencia corre distribuida sobre Spark aunque sea 1 fila:
# el plan de ejecución pasa por el mismo DAG que en batch.
 
def inferir_kmeans():
    log_rc = float(dbutils.widgets.get("log_review_count"))
    avg_rt = float(dbutils.widgets.get("avg_rating"))
    std_rt = float(dbutils.widgets.get("std_rating"))
    log_wc = float(dbutils.widgets.get("log_avg_word_count"))
 
    # Carga el pipeline entrenado desde el volumen Unity Catalog
    modelo = PipelineModel.load(f"{VOLUMEN_MODELOS}/modelo_kmeans_usuarios")
 
    # Crear DataFrame distribuido con los features del usuario
    df = spark.createDataFrame([{
        "log_review_count":   log_rc,
        "avg_rating":         avg_rt,
        "std_rating":         std_rt,
        "log_avg_word_count": log_wc,
    }])
 
    # .transform() aplica: VectorAssembler → StandardScaler → KMeans.predict
    resultado  = modelo.transform(df)
    cluster_id = resultado.select("cluster").collect()[0][0]
    nombre     = NOMBRES_CLUSTERS.get(cluster_id, f"Cluster {cluster_id}")
 
    return json.dumps({
        "tipo":    "kmeans",
        "cluster": cluster_id,
        "nombre":  nombre,
    })

In [0]:
# ── Rama LDA ─────────────────────────────────────────────────
# Carga el PipelineModel NLP completo (Tokenizer → StopWordsRemover →
# CountVectorizer → IDF) y el LocalLDAModel por separado porque fueron
# guardados en rutas distintas en el notebook de entrenamiento.
# El texto pasa por el mismo preprocesamiento que el corpus de entrenamiento.
 
def inferir_lda():
    texto_raw    = dbutils.widgets.get("texto")
    texto_limpio = re.sub(r"[^a-záéíóúñ0-9\s]", "", texto_raw.lower())
    texto_limpio = re.sub(r"\s+", " ", texto_limpio).strip()
 
    # Pipeline NLP: Tokenizer → StopWordsRemover → CountVectorizer → IDF
    modelo_nlp = PipelineModel.load(f"{VOLUMEN_MODELOS}/modelo_nlp_tfidf")
    # LocalLDAModel guardado por separado (optimizer='online' produce LocalLDAModel)
    modelo_lda = LocalLDAModel.load(f"{VOLUMEN_MODELOS}/modelo_lda")
 
    # Crear DataFrame con el texto limpio
    df = spark.createDataFrame([(texto_limpio,)], schema=["caption_clean"])
 
    # Aplicar pipeline NLP completo — produce columna tf_vector y tfidf_vector
    df = modelo_nlp.transform(df)
 
    # LDA usa featuresCol='tf_vector' (igual que en entrenamiento)
    df = modelo_lda.transform(df)
 
    dist   = df.select("topicDistribution").collect()[0][0]
    tid    = int(np.argmax(dist.toArray()))
    nombre = NOMBRES_TOPICOS.get(tid, f"Tópico {tid}")
 
    return json.dumps({
        "tipo":   "lda",
        "topico": tid,
        "nombre": nombre,
    })

In [0]:
if modo == "kmeans":
    dbutils.notebook.exit(inferir_kmeans())
elif modo == "lda":
    dbutils.notebook.exit(inferir_lda())
else:
    dbutils.notebook.exit(json.dumps({"error": f"modo desconocido: {modo}"}))